# ASTraM Event Forecasting - OSM Enrichment (Optional Stretch Goal)

This notebook demonstrates how to enrich the tabular incident data with OpenStreetMap road classifications (e.g., primary, residential) by performing a spatial nearest-neighbor join.

In [1]:
import pandas as pd
import geopandas as gpd
import warnings
warnings.filterwarnings('ignore')

## 1. Load the Data
We load the `cleaned_astram_events.csv` which contains the incidents with their `latitude` and `longitude`.

In [2]:
df = pd.read_csv('../data/cleaned_astram_events.csv')
print(f"Incident Data Shape: {df.shape}")

Incident Data Shape: (7116, 44)


## 2. Load and Filter OSM Road Network
We load `export.geojson` and filter out non-vehicular paths like footways and steps.

In [3]:
# Load OSM roads
roads = gpd.read_file('../export.geojson')

# Filter to vehicular classes only
vehicle_classes = [
    "motorway", "trunk", "primary", "secondary", "tertiary",
    "unclassified", "residential", "living_street",
    "motorway_link", "trunk_link", "primary_link", "secondary_link", "tertiary_link",
]
roads = roads[roads["highway"].isin(vehicle_classes)].copy()

print(f"Filtered Roads Shape: {roads.shape}")
print(f"Fraction of roads with 'lanes' tag: {roads['lanes'].notna().mean():.2f}")

Filtered Roads Shape: (109220, 371)
Fraction of roads with 'lanes' tag: 0.06


## 3. Spatial Nearest-Neighbor Join
We convert both the incidents and roads to a local metric CRS (EPSG:32643 for Bengaluru) to compute accurate distances, then join each incident to the nearest road linestring.

In [4]:
# Reproject roads to metric CRS (UTM Zone 43N)
roads_m = roads.to_crs(epsg=32643)

# Convert incidents to a GeoDataFrame and reproject
incidents = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326",
).to_crs(epsg=32643)

# Perform the nearest-road join
enriched = gpd.sjoin_nearest(
    incidents,
    roads_m[["highway", "lanes", "geometry"]],
    how="left",
    distance_col="dist_to_road_m",
)

# Since multiple roads could be at the same distance, we drop duplicates to keep 1:1 mapping
enriched = enriched[~enriched.index.duplicated(keep='first')]

df["osm_highway_class"] = enriched["highway"]
df["osm_lanes"] = enriched["lanes"]
df["dist_to_nearest_road_m"] = enriched["dist_to_road_m"]

print("Enrichment Complete!")
print(df[['id', 'osm_highway_class', 'osm_lanes', 'dist_to_nearest_road_m']].head())

Enrichment Complete!
           id osm_highway_class osm_lanes  dist_to_nearest_road_m
0  FKID000000          tertiary       NaN                2.577087
1  FKID000001         secondary         4               12.635905
2  FKID000002         secondary         3                3.736194
3  FKID000003         secondary         2                3.713579
4  FKID000004           primary       NaN                5.278644


## 4. Export Enriched Data

In [5]:
df.to_csv('../data/enriched_astram_events.csv', index=False)
print("Saved enriched dataset to data/enriched_astram_events.csv")

Saved enriched dataset to data/enriched_astram_events.csv
